# Tiny LLM Decoder 3 (Pytorch End to End)

In the third version we are implementing Pytorch end to end.

In [1]:
import numpy as np
import scipy.stats as stats
import torch
import torch.nn as nn
from torch.nn import functional as F

## LLM Architecture

At high level, the training process is as follows:

```text

Text_processing -> Transformer Network -> Compare Loss -> Back Propagation
```

The inference process is as follows:

```text
Text_processing -> Transformer Network -> Predict Next Token -> Repeat until token size reached
```

### Corpus

In [2]:
# Text corpus
corpus = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""


### Parameters

In [3]:
# Set Parameters

# random seed
random_seed = 42
torch.manual_seed(random_seed)

# d_model is embedding dimension
d_model = 4

# block_size is the same as context window or maximum sequence length
block_size = 5



## Text Processing - Tokenization

In [4]:
# Tokenization

# Split corpus into words
word_tokens = corpus.split()

# Use set to isolate unique words
vocab = sorted(list(set(word_tokens)))
vocab_size = len(vocab)

# Create token id and create lookup table from word to token id
# i is the index or tken id in this case, so we assign token id using sequential number
words_to_id_ref = {word: i for i, word in enumerate(vocab)}

# Create lookup table from id to words
id_to_words_ref = {i: word for i, word in enumerate(vocab)}

In [5]:
vocab_size

19

In [6]:
# convert corpus to id
corpus_in_id = [words_to_id_ref[word] for word in word_tokens]


### Function to extract blocks

In [7]:
def get_sequence_block(block_size, start_position, corpus_in_id):
  """
  This function extract the block in id from the corpus in id
  User need to supply the block size and the start position
  Return: X, Y, next_start_position
  """
  X = corpus_in_id[start_position: start_position+block_size]
  Y = corpus_in_id[start_position+1: start_position+block_size+1]
  next_start_position = start_position + block_size
  return X, Y, next_start_position

In [8]:
def get_N(corpus_in_id, block_size):
  """
  This function calculate the number of block we need to run for the entire corpus
  """
  N = (len(corpus_in_id)-1) // block_size
  return N

### Initialization of Models


In [9]:
# Setup embeddings initialized randomly
embeddings = torch.nn.Embedding(vocab_size, d_model)
embeddings


Embedding(19, 4)

In [10]:
# Initializing weights (4,4) for WQ, WK and WV
WQ = nn.Parameter(torch.randn(d_model, d_model))
WK = nn.Parameter(torch.randn(d_model, d_model))
WV = nn.Parameter(torch.randn(d_model, d_model))

In [11]:
ffn = nn.Sequential(
    nn.Linear(d_model, d_model, bias=False),
    nn.ReLU(),
    nn.Linear(d_model, d_model, bias=False)
)

In [12]:
norm1 = nn.LayerNorm(d_model)
norm2 = nn.LayerNorm(d_model)

In [13]:
parameters = (
    list(embeddings.parameters())
    + [WQ, WK, WV]
    + list(ffn.parameters())
    + list(norm1.parameters())
    + list(norm2.parameters())
)

print(sum(p.numel() for p in parameters))
print(all(p.requires_grad for p in parameters))

172
True


### First Pass

In [14]:
N = get_N(corpus_in_id, block_size=block_size)
N

6

In [15]:
X_id, Y_id, next_start = get_sequence_block(block_size=block_size, start_position=0, corpus_in_id=corpus_in_id)
X_id, Y_id, next_start

([10, 6, 0, 9, 8], [6, 0, 9, 8, 7], 5)

In [16]:
X_tensor = torch.tensor(X_id, dtype=torch.long)
Y_tensor = torch.tensor(Y_id, dtype=torch.long)

In [17]:
def forward(X):
    # Word embeddings + positional encoding
    X_input = embeddings(X)
    position = torch.arange(len(X), dtype=X_input.dtype).reshape(-1, 1)
    X = X_input + position

    # Q, K, V projections
    Q = X @ WQ
    K = X @ WK
    V = X @ WV

    # Scaled dot-product attention
    attention_score = Q @ K.T
    stable_factor = np.sqrt(d_model)
    attention_stable = attention_score / stable_factor

    # Causal masking + softmax
    causal_mask = torch.triu(
        torch.full_like(attention_stable, float("-inf")),
        diagonal=1
    )
    attention_stable_mask = attention_stable + causal_mask
    attention_softmax = torch.softmax(attention_stable_mask, dim=-1)

    # Attention output + residual + norm1
    attention_output = attention_softmax @ V

    # Residual Layer
    attention_output_residual = attention_output + X_input

    # norm1 layer
    attention_output_residual_scaled = norm1(attention_output_residual)

    # FFN + residual + norm2
    ffn_1 = ffn(attention_output_residual_scaled)

    # Residual 2 Layer
    seond_residual = ffn_1 + attention_output_residual_scaled

    # Norm 2 Layer
    transformer_output = norm2(seond_residual)

    # Vocabulary projection
    logits = transformer_output @ embeddings.weight.T

    return logits

In [18]:
logits = forward(X_tensor)
logits.shape

torch.Size([5, 19])

In [19]:
print(logits.shape)
print(logits.requires_grad)

torch.Size([5, 19])
True


In [20]:
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, Y_tensor)

In [21]:
print(loss)
print(loss.shape)
print(loss.requires_grad)

tensor(4.5187, grad_fn=<NllLossBackward0>)
torch.Size([])
True


In [22]:
loss.backward()

In [23]:
print(WQ.grad)
print(WK.grad)
print(WV.grad)

tensor([[-0.0119,  0.0007, -0.0001, -0.0014],
        [-0.0101,  0.0007,  0.0009, -0.0007],
        [-0.0098,  0.0002, -0.0023, -0.0022],
        [-0.0026, -0.0008, -0.0060, -0.0032]])
tensor([[-0.0027, -0.0133,  0.0029,  0.0068],
        [-0.0018, -0.0121,  0.0029,  0.0065],
        [-0.0022, -0.0058,  0.0007,  0.0023],
        [ 0.0036,  0.0272, -0.0067, -0.0148]])
tensor([[ 0.0030,  0.0715,  0.0485, -0.1230],
        [-0.0576,  0.0447,  0.0862, -0.0733],
        [ 0.0268,  0.0637,  0.0805, -0.1710],
        [ 0.0289, -0.0099, -0.0026, -0.0163]])


In [24]:
print(embeddings.weight.grad.shape)

torch.Size([19, 4])


In [25]:
for i, p in enumerate(parameters):
    print(i, p.shape, p.grad is not None)

0 torch.Size([19, 4]) True
1 torch.Size([4, 4]) True
2 torch.Size([4, 4]) True
3 torch.Size([4, 4]) True
4 torch.Size([4, 4]) True
5 torch.Size([4, 4]) True
6 torch.Size([4]) True
7 torch.Size([4]) True
8 torch.Size([4]) True
9 torch.Size([4]) True


In [26]:
print("WQ:", WQ.grad is not None)
print("WK:", WK.grad is not None)
print("WV:", WV.grad is not None)

print("FFN:", [p.grad is not None for p in ffn.parameters()])
print("Norm1:", [p.grad is not None for p in norm1.parameters()])
print("Norm2:", [p.grad is not None for p in norm2.parameters()])

WQ: True
WK: True
WV: True
FFN: [True, True]
Norm1: [True, True]
Norm2: [True, True]


In [27]:
optimizer = torch.optim.SGD(parameters, lr=0.01)

In [28]:
optimizer.step()

In [29]:
new_logits = forward(X_tensor)
new_loss = criterion(new_logits, Y_tensor)

print("New loss:", new_loss.item())

New loss: 4.47305965423584


## Training Loop

In [30]:
def llm_train(X_tensor, Y_tensor, iters=10):

  for i in range(iters):
    optimizer.zero_grad()

    logits = forward(X_tensor)
    loss = criterion(logits, Y_tensor)
    print(f"iters {i} : Loss = {loss.item():.4f}")

    loss.backward()
    optimizer.step()

In [31]:
llm_train(X_tensor, Y_tensor, iters=1000)

iters 0 : Loss = 4.4731
iters 1 : Loss = 4.4286
iters 2 : Loss = 4.3853
iters 3 : Loss = 4.3432
iters 4 : Loss = 4.3021
iters 5 : Loss = 4.2620
iters 6 : Loss = 4.2229
iters 7 : Loss = 4.1848
iters 8 : Loss = 4.1475
iters 9 : Loss = 4.1111
iters 10 : Loss = 4.0755
iters 11 : Loss = 4.0408
iters 12 : Loss = 4.0068
iters 13 : Loss = 3.9735
iters 14 : Loss = 3.9410
iters 15 : Loss = 3.9092
iters 16 : Loss = 3.8781
iters 17 : Loss = 3.8476
iters 18 : Loss = 3.8178
iters 19 : Loss = 3.7885
iters 20 : Loss = 3.7599
iters 21 : Loss = 3.7319
iters 22 : Loss = 3.7045
iters 23 : Loss = 3.6776
iters 24 : Loss = 3.6513
iters 25 : Loss = 3.6255
iters 26 : Loss = 3.6002
iters 27 : Loss = 3.5755
iters 28 : Loss = 3.5512
iters 29 : Loss = 3.5274
iters 30 : Loss = 3.5041
iters 31 : Loss = 3.4813
iters 32 : Loss = 3.4589
iters 33 : Loss = 3.4369
iters 34 : Loss = 3.4154
iters 35 : Loss = 3.3943
iters 36 : Loss = 3.3737
iters 37 : Loss = 3.3534
iters 38 : Loss = 3.3335
iters 39 : Loss = 3.3141
iters 40 :

In [32]:
with torch.no_grad():
    logits = forward(X_tensor)
    predicted_ids = torch.argmax(logits, dim=-1)

print(predicted_ids)
print(Y_tensor)

tensor([6, 0, 9, 8, 7])
tensor([6, 0, 9, 8, 7])


We use argmax to get the highest probability and the prediction is good.

## END